# 端到端示例 2：Reaction-to-Purification

目标：完成反应、建立相系统、萃取/分相、洗涤/干燥/浓缩，最后 final assay。

关键点：下游纯化不能在 `terminate` 之后进行；当前 schema 要先 `add_phase`，再 `add_extractant`。

## 1. 任务规划与动作空间

先检查 task contract，确认允许 separation operations。

In [ ]:
from __future__ import annotations

from pathlib import Path

import gymnasium as gym
import pandas as pd
from IPython.display import display

import chemworld  # noqa: F401 - registers ChemWorld

OUTPUT_DIR = Path("runs/end_to_end")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.precision", 4)

env = gym.make("ChemWorld", task_id="reaction-to-purification", seed=0)
_obs, _info = env.reset(seed=0)
display(env.unwrapped.task_prompt())
display(pd.DataFrame(env.unwrapped.available_actions()).head(16))
display(env.unwrapped.action_schema("add_extractant"))
display(env.unwrapped.action_schema("separate_phase"))
env.close()


## 2. 合法 reaction-to-purification recipe

流程顺序：反应生成产物 -> 淬灭 -> 建立有机相 -> 萃取混合静置 -> 分相 -> 洗涤干燥浓缩 -> 终止 -> final assay。

In [ ]:
recipe = [
    {"operation": "add_solvent", "volume_L": 0.030, "solvent": 1},
    {"operation": "add_reagent", "amount_mol": 0.012},
    {"operation": "add_catalyst", "catalyst": 2, "catalyst_amount_mol": 0.0004},
    {"operation": "heat", "target_temperature_K": 350.0, "duration_s": 1200.0, "stirring_speed_rpm": 800.0},
    {"operation": "quench"},
    {"operation": "add_phase", "phase": "organic", "volume_L": 0.020},
    {"operation": "add_extractant", "extractant": "organic", "volume_L": 0.010},
    {"operation": "mix", "duration_s": 120.0, "stirring_speed_rpm": 600.0},
    {"operation": "settle", "duration_s": 300.0},
    {"operation": "separate_phase", "target_phase": "organic"},
    {"operation": "wash", "wash_volume_L": 0.010},
    {"operation": "dry"},
    {"operation": "concentrate", "duration_s": 300.0},
    {"operation": "terminate"},
    {"operation": "measure", "instrument": "final_assay"},
]


## 3. 执行、验证和指标表

重点看 purification score、purity、recovery、process mass balance 和 precondition failure。

In [ ]:
def run_recipe(task_id: str, recipe: list[dict], *, seed: int = 0) -> tuple[pd.DataFrame, dict]:
    env = gym.make("ChemWorld", task_id=task_id, seed=seed)
    _obs, info = env.reset(seed=seed)
    rows = []
    final_info = info
    try:
        for step, action in enumerate(recipe, start=1):
            validation = env.unwrapped.validate_action(action)
            _obs, reward, terminated, truncated, info = env.step(action)
            final_info = info
            rows.append(
                {
                    "step": step,
                    "operation": action["operation"],
                    "valid_before_step": validation["valid"],
                    "invalid_reasons": "; ".join(validation.get("invalid_reasons", [])),
                    "reward": float(reward),
                    "leaderboard_score": info.get("leaderboard_score"),
                    "precondition_failed": info.get("constraint_flags", {}).get("precondition_failed"),
                    "unsafe": info.get("constraint_flags", {}).get("unsafe"),
                    "cost": info.get("cost"),
                    "observed_keys": ", ".join(info.get("observed_keys", [])),
                    "has_raw_signal": bool(info.get("raw_signal")),
                    "terminated": terminated,
                    "truncated": truncated,
                }
            )
            if terminated or truncated:
                break
    finally:
        env.close()
    return pd.DataFrame(rows), final_info


def spectra_frame(final_info: dict, channel: str = "hplc") -> pd.DataFrame:
    packet = final_info.get("raw_signal", {})
    spectra = packet.get("spectra", {}) if isinstance(packet, dict) else {}
    signal = spectra.get(channel, {})
    if channel in {"hplc", "gc"}:
        return pd.DataFrame({"x": signal.get("time_min", []), "signal": signal.get("intensity", [])})
    if channel == "uvvis":
        return pd.DataFrame({"x": signal.get("wavelength_nm", []), "signal": signal.get("absorbance", [])})
    return pd.DataFrame()

rows, final_info = run_recipe("reaction-to-purification", recipe, seed=0)
display(rows)
display(final_info.get("processed_estimate", {}))
rows.to_csv(OUTPUT_DIR / "reaction_to_purification_trace.csv", index=False)
assert not rows["precondition_failed"].any(), "recipe should be valid under current task contract"


## 4. 谱图与纯化权衡

比较 HPLC 主峰、impurity signal、purity/recovery。高纯度但低 recovery 不是一定更好。

In [ ]:
hplc = spectra_frame(final_info, "hplc")
if not hplc.empty:
    hplc.plot(x="x", y="signal", title="Purified final assay HPLC", xlabel="time / min")
metrics = final_info.get("processed_estimate", {})
tradeoff = pd.DataFrame([{
    "leaderboard_score": final_info.get("leaderboard_score"),
    "purity": metrics.get("purity"),
    "recovery": metrics.get("recovery"),
    "yield": metrics.get("yield"),
    "process_mass_balance_error": metrics.get("process_mass_balance_error"),
}])
display(tradeoff)


## 5. 反思记录

请用化工语言解释：分相和浓缩提高了什么，牺牲了什么。

In [ ]:
reflection = {
    "best_score": rows["leaderboard_score"].dropna().max(),
    "purification_hypothesis": "示例：有机相分离提高了可检测纯度，但 recovery 仍低，下一轮应比较相体积和 target_phase。",
    "next_experiment": {"add_phase_volume_L": 0.030, "target_phase": "organic", "reason": "提高分配和回收"},
    "evidence": str(OUTPUT_DIR / "reaction_to_purification_trace.csv"),
}
reflection
